# pymefpp_partition_data

Notebook to create a partition of the calibration files for later reduction.

The user needs to indicate the following directories:

 - raw_data_dir: directory where the raw data is located. Typically outside the project_dir.
 - project_dir: directory where the project reduction will be done. It will contain the reduce_calib_dir and the reduced_science_dir
 - reduced_calib_dir: directory where the reduced calibrations will be placed
 - reduced_science_dir: directory root where the data reduction arena will be established.
 
This is the first notebook within the mefpp series, where each one is numbered from 01 to 07, indicating the order in which they must be run.

# Testing data

Notebook 1 run without any troubles.

In [ ]:
# importing needed system modules
import glob
import sys
import os
from pathlib import Path
import logging

# here we import the local mefpp_run library
# where the python scripts are
sys.path.insert(0, os.getcwd())
import mefpp_associate_reduction as mar
import pycavi_tools
import pycavi_instrument

In [ ]:
# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# we set location of the project configuration file and
# we get the basic project info from the user with a tkinter widgets window
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']


# Should get these from an instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir='./config/', instrument='OCAM')
file_prefix = instrument.config.get('Data', 'raw_file_prefix')
file_suffix = instrument.config.get('Data', 'raw_file_suffix')

# create a list with all the images in the raw data dir
# glob('src/**/*.c', recursive=True)
print("argument for glob.glob: {}".format(raw_data_dir + "/**/"+file_prefix+"*"+file_suffix))
listing = glob.glob(raw_data_dir + "/**/"+file_prefix+"*"+file_suffix, recursive=True)
print("number of raw files found: {}".format(len(listing)))

In [ ]:
# create the root dir if not exist, then all the rest of folders
try:
    os.makedirs(reduced_cal_dir, exist_ok=True)
except Exception as e:
    print(e)
    pass

try:
    os.makedirs(reduced_science_dir, exist_ok=True)
    #os.makedirs(reduced_science_dir+'/logs/')
except Exception as e:
    print(e)
    pass
#create the log_dir outside the science dir.
try:
    os.makedirs(log_dir, exist_ok=True)
except Exception as e:
    print(e)
    pass

# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_partition_data.log"),
                              logging.StreamHandler()
                    ])
    

In [ ]:
# we find all OMEGACAM fits files below the raw data directory
logging.info('%d OmegaCAM frames in %s'%(len(listing), raw_data_dir))

exposurePartition = mar.PartitionsTree()


i_exposure = 0

for file_name in listing:

    i_exposure+= 1
    exposure = mar.Exposure(file_name)
    exposurePartition.insertExposure(exposure)
    biasDict = {k:v for (k,v) in exposurePartition.partList.items() if 'bias/' in k}
    domeDict = {k:v for (k,v) in exposurePartition.partList.items() if 'dome/' in k}
    skyDict = {k:v for (k,v) in exposurePartition.partList.items() if 'sky/' in k}
    stdDict = {k:v for (k,v) in exposurePartition.partList.items() if 'std/' in k}
    sciDict = {k:v for (k,v) in exposurePartition.partList.items() if 'science/' in k}
    if i_exposure%10:
        Total = len(biasDict) + len(domeDict) + \
                len(skyDict) + len(stdDict) +\
                len(sciDict)
        print("BIAS:     %d" % len(biasDict), \
                          "DOME:     %d" % len(domeDict), \
                          "SKY:      %d" % len(skyDict), \
                          "STD:      %d" % len(stdDict), \
                          "SCIENCE:  %d" % len(sciDict), \
                          "Total     %d" % Total, end='\r')


In [ ]:
# we create directory structure to reduce the calibration data

# bad pixel masks BPM are needed but static at this point we create
# a directory to contain them
try:
    os.makedirs(static_cal_dir+'bpm')
except:
    pass

Path(static_cal_dir+'bpm'+'/bpm.list').touch()
Path(static_cal_dir+'bpm'+'/toreduce_bpm').touch()


In [ ]:
# here we create a bias.list file in each bias partition
mar.createFrameList(reduced_cal_dir, 'bias.list', biasDict)

In [ ]:
# here we create a dome.list file in dome partition
mar.createFrameList(reduced_cal_dir, 'dome.list', domeDict)

In [ ]:
# here we create a sky.list file in each sky partition
mar.createFrameList(reduced_cal_dir, 'sky.list', skyDict)

In [ ]:
# here we create a science.list file in each science partition
mar.createFrameList(reduced_science_dir, 'science.list', sciDict)